### Download the negative tsv file from Uniprot using request API

In [31]:
import requests
import pandas as pd

url = "https://rest.uniprot.org/uniprotkb/stream"  # endpoint dell'API UniProt

# Parametri della query, passati come dizionario
params = {
    "query": "(reviewed:true) AND (fragment:false) AND (taxonomy_id:2759) "
             "AND (length:[40 TO *]) AND "
             "((cc_scl_term_exp:SL-0091) OR (cc_scl_term_exp:SL-0188) OR "
             "(cc_scl_term_exp:SL-0173) OR (cc_scl_term_exp:SL-0209) OR "
             "(cc_scl_term_exp:SL-0204) OR (cc_scl_term_exp:SL-0039)) "
             "NOT (ft_signal:*)",          # la query di ricerca su UniProt
    "fields": "accession,reviewed,id,length,ft_transmem,lineage,sequence",  # colonne da scaricare
    "format": "tsv",   # formato di output: tab-separated values
}

r = requests.get(url, params=params, timeout=300)
r.raise_for_status()

# salva il file scaricato su disco
with open("uniprot_results.tsv", "w") as f:
    f.write(r.text)

# rilegge il file dal disco
df = pd.read_csv("uniprot_results.tsv", sep="\t")


In [32]:
print(df.info)

<bound method DataFrame.info of             Entry  Reviewed   Entry Name  Length  \
0      A0A061ACU2  reviewed  PIEZ1_CAEEL    2442   
1      A0A067XGX8  reviewed  AROG2_PETHY     512   
2      A0A067XH53  reviewed  AROG1_PETHY     533   
3      A0A075D657  reviewed  PINMT_VINMI     322   
4      A0A075TRC0  reviewed   PATK_PENEN    1776   
...           ...       ...          ...     ...   
18974      Q9TKX7  reviewed  YCF81_NEPOL     138   
18975      Q9UT54  reviewed   YI95_SCHPO     125   
18976      Q9UTM1  reviewed   YIV1_SCHPO     112   
18977      Q9XPS5  reviewed  YCF70_WHEAT      42   
18978      Q9Y807  reviewed   YN92_SCHPO     263   

                                           Transmembrane  \
0      TRANSMEM 6..26; /note="Helical; Name=1"; /evid...   
1                                                    NaN   
2                                                    NaN   
3                                                    NaN   
4                                          

### Filter the entries of the negative set 

In [33]:
import re
import pandas as pd

def tm_starts(s):
    if pd.isna(s):
        return []
    return [int(x) for x in re.findall(r'TRANSMEM\s+[<>?]?(\d+)', str(s))]

def parse_lineage(s):
    # es: 'cellular organisms (no rank), Eukaryota (domain), ..., Caenorhabditis elegans (species)'
    return [p.strip() for p in str(s).split(',')]

def kingdom(s):
    parts = parse_lineage(s)
    names = {re.sub(r'\s*\(.*\)$', '', p) for p in parts}
    if 'Metazoa' in names:         return 'Metazoa'
    if 'Fungi' in names:           return 'Fungi'
    if 'Viridiplantae' in names:   return 'Plants'
    return 'Other'

def organism(s):
    parts = parse_lineage(s)
    sp = [p for p in parts if p.endswith('(species)')]
    last = sp[-1] if sp else parts[-1]
    return re.sub(r'\s*\(.*\)$', '', last)

out = pd.DataFrame({
    'accession':      df['Entry'],
    'organism':       df['Taxonomic lineage'].apply(organism),
    'kingdom':        df['Taxonomic lineage'].apply(kingdom),
    'length':         df['Length'].astype(int),
    'tm_in_first_90': df['Transmembrane'].apply(lambda s: any(p <= 90 for p in tm_starts(s))),
})

out.to_csv('negative_data.tsv', sep='\t', index=False)

In [34]:
df2 = pd.read_table('negative_data.tsv')

In [35]:
df2.head(10)

,accession,organism,kingdom,length,tm_in_first_90
0,A0A061ACU2,Caenorhabditis,Metazoa,2442,True
1,A0A067XGX8,Petunia,Plants,512,False
2,A0A067XH53,Petunia,Plants,533,False
3,A0A075D657,Vinca,Plants,322,False
4,A0A075TRC0,Penicillium,Fungi,1776,False
5,A0A076FFM5,Ocimum,Plants,523,False
6,A0A078CGE6,Brassica,Plants,1299,False
7,A0A087X1C5,Homo,Metazoa,515,True
8,A0A095C325,Cryptococcus deuterogattii,Fungi,1408,False
9,A0A096LP01,Homo,Metazoa,95,True


In [30]:
print(df2['tm_in_first_90'].value_counts())

tm_in_first_90
False    14970
True      4009
Name: count, dtype: int64


### Create filtered fasta file

In [ ]:
with open("negative_SP_clean.fasta", "w") as f:
    for _, row in df.iterrows():
        f.write(f">{row['Entry']}\n{row['Sequence']}\n")   